# Relatório de vendas — do banco de dados até o site

Este notebook transforma o dump do banco de dados em 3 arquivos que, juntos, formam o site:

- `data.json` — os números já calculados (receita, pedidos, produtos, pagamentos, máquinas)
- `index.html` — versão leve, para abrir no navegador com servidor ou no GitHub Pages
- `index-standalone.html` — versão com tudo dentro de 1 arquivo só, para mandar por WhatsApp ou e-mail e abrir com dois cliques

## Passo a passo

**Passo 1 — Subir o dump**
Você sobe o arquivo `.sql` que saiu do SSMS. É o dump da camada Gold, com as 5 tabelas: `dim_evento`, `dim_maquinas`, `dim_produtos`, `fact_vendas` e `fact_pagamentos`.

**Passo 2 — Ler os dados**
O notebook lê o `.sql` e separa as tabelas. Nada é alterado no banco, só lido.

**Passo 3 — Conferir se está tudo certo**
Antes de calcular qualquer coisa, o notebook confere se as tabelas vieram completas e se os números batem. Se alguma conferência falhar, ele para aqui. É melhor conferir agora do que descobrir depois que o site ficou errado.

**Passo 4 — Montar o `data.json`**
Com os dados conferidos, o notebook calcula os totais e monta o `data.json`. Vendas canceladas e devoluções ficam fora da receita e aparecem separadas, lá embaixo do site.

**Passo 5 — Gerar o site**
Você sobe o `index.html` atual (a versão leve ou a standalone, tanto faz). O notebook recria a leve e a standalone a partir do mesmo `data.json`, com os mesmos dados, gráficos e visual.

**Passo 6 — Baixar os arquivos**
Você baixa os 3 arquivos atualizados: `data.json`, `index.html` e `index-standalone.html`.

> Rode as células na ordem, de cima para baixo, uma por uma, vendo o resultado de cada uma.


In [ ]:
# CÉLULA 1 — Subir o arquivo de dados (.sql do SSMS)
# O que esta célula faz: pede para você escolher o arquivo .sql que saiu do SSMS.
# Por que: sem esse arquivo, não há o que calcular. Ele tem as 5 tabelas da camada Gold.
from google.colab import files
import pathlib

print("Escolha o arquivo .sql do dump (ex: script.sql)...")
up = files.upload()
# Pega o primeiro arquivo .sql que foi enviado
sql_nome = next((n for n in up.keys() if n.lower().endswith(".sql")), None)
if sql_nome is None:
    raise ValueError("Nenhum arquivo .sql encontrado. Suba de novo o dump do SSMS.")
# Salva como script.sql para as próximas células usarem sempre o mesmo nome
pathlib.Path("/content/script.sql").write_bytes(up[sql_nome])
print(f"Arquivo recebido: {sql_nome} -> salvo como /content/script.sql")


In [ ]:
# CÉLULA 2 — Ler os dados do .sql
# O que esta célula faz: abre o .sql e separa as 5 tabelas, linha por linha.
# Por que: o .sql é um texto grande com INSERTs. Precisamos transformar esse texto em tabelas
# para conseguir somar receita, contar pedidos e montar os gráficos.
import re, json
from pathlib import Path

# --- Leitura do arquivo ---
# O SSMS pode salvar em codificações diferentes. Tentamos as mais comuns até uma funcionar.
def read_sql(path: Path) -> str:
    raw = path.read_bytes()
    for enc in ("utf-16", "utf-8-sig", "utf-8", "latin-1"):
        try:
            return raw.decode(enc)
        except (UnicodeDecodeError, UnicodeError):
            continue
    return raw.decode("utf-8", errors="ignore")

# --- Leitura de valores dentro de um INSERT ---
# Uma string vem entre aspas simples. Duas aspas juntas ('') significam uma aspa dentro do texto.
def parse_string(s: str, i: int):
    j = i + 1; buf = []
    while j < len(s):
        if s[j] == "'":
            if j + 1 < len(s) and s[j + 1] == "'":
                buf.append("'"); j += 2; continue
            return "".join(buf), j + 1
        buf.append(s[j]); j += 1
    raise ValueError("String sem fechamento")

def parse_values(s: str):
    vals = []; i = 0
    while i < len(s):
        while i < len(s) and s[i] in " \t,": i += 1
        if i >= len(s): break
        if s.startswith("NULL", i):
            vals.append(None); i += 4
        elif s.startswith("CAST(", i):
            depth = 0; j = i
            while j < len(s):
                if s[j] == "'":
                    _, j = parse_string(s, j); continue
                if s[j] == "(": depth += 1
                elif s[j] == ")":
                    depth -= 1
                    if depth == 0: break
                j += 1
            inner = s[i + 5:j].strip()
            if inner.startswith("N'") or inner.startswith("'"):
                v, _ = parse_string(inner, 1 if inner.startswith("N'") else 0)
                vals.append(v)
            else:
                num = re.match(r"-?[0-9.]+", inner).group(0)
                vals.append(float(num) if "." in num else int(num))
            i = j + 1
        elif s[i] == "N" and i + 1 < len(s) and s[i + 1] == "'":
            v, j = parse_string(s, i + 1); vals.append(v); i = j
        elif s[i] == "'":
            v, j = parse_string(s, i); vals.append(v); i = j
        else:
            m = re.match(r"-?[0-9.]+", s[i:])
            if not m: raise ValueError(f"Valor inesperado em: {s[i:i+30]!r}")
            num = m.group(0); vals.append(float(num) if "." in num else int(num)); i += len(num)
    return vals

# --- Leitura de uma tabela inteira ---
TABLES = {
    "dim_evento": ["id_evento", "data", "horario_inicio", "tipo_evento"],
    "dim_maquinas": ["dispositivo_id", "dispositivo_clover"],
    "dim_produtos": ["produto_id", "produto", "categoria"],
    "fact_vendas": ["id_evento", "nome_comercio", "numero_pedido", "hora", "total", "produto_id", "quantidade", "preco_unitario", "total_produto", "metodo_pagamento", "dispositivo_id", "status_venda", "estado"],
    "fact_pagamentos": ["pagamento_id", "numero_pedido", "metodo_pagamento", "bandeira", "valor_pagamento"],
}

def parse_table(text: str, table: str):
    cols = TABLES[table]
    pat = re.compile(r"INSERT \[gold\]\.\[" + table + r"\].*?VALUES\s*", re.I | re.S)
    rows = []
    for m in pat.finditer(text):
        i = m.end()
        while i < len(text) and text[i] in " \t\n\r": i += 1
        if i >= len(text) or text[i] != "(": continue
        depth = 0; j = i; in_str = False
        while j < len(text):
            c = text[j]
            if in_str:
                if c == "'":
                    if j + 1 < len(text) and text[j + 1] == "'": j += 2; continue
                    in_str = False
            else:
                if c == "'": in_str = True
                elif c == "(": depth += 1
                elif c == ")":
                    depth -= 1
                    if depth == 0: break
            j += 1
        inner = text[i + 1:j]
        vals = parse_values(inner)
        if len(vals) != len(cols):
            raise ValueError(f"{table}: esperado {len(cols)} valores, veio {len(vals)}")
        rows.append(dict(zip(cols, vals)))
    return rows

def to_num(v, default=0.0):
    # Transforma um valor em número. Se vier vazio ou texto, usa o padrão.
    try:
        if v is None or v == "": return default
        return float(v)
    except (TypeError, ValueError):
        return default

# --- Executa a leitura ---
text = read_sql(Path("/content/script.sql"))
tables = {t: parse_table(text, t) for t in TABLES}
for t, rows in tables.items():
    print(f"{t}: {len(rows)} linhas")


In [ ]:
# CÉLULA 3 — Conferências rápidas (se alguma falhar, pare aqui)
# O que esta célula faz: confere se as tabelas vieram completas e se os números fazem sentido.
# Por que: se o dump veio incompleto ou com a coluna nova vazia, é melhor descobrir agora,
# antes de calcular o JSON e gerar o site.
import collections

def conferir(cond, msg):
    # Se a condição for falsa, para o notebook aqui com a mensagem.
    if not cond:
        raise ValueError("CONFERÊNCIA FALHOU: " + msg)
    print("OK — " + msg)

# 1. Todas as tabelas têm linhas?
for t in ["dim_evento", "dim_maquinas", "dim_produtos", "fact_vendas", "fact_pagamentos"]:
    conferir(len(tables[t]) > 0, f"{t} tem linhas ({len(tables[t])})")

# 2. Todo produto e toda máquina da venda existe nas tabelas de apoio?
prod_ids = {r["produto_id"] for r in tables["dim_produtos"]}
maq_ids = {r["dispositivo_id"] for r in tables["dim_maquinas"]}
ev_ids = {r["id_evento"] for r in tables["dim_evento"]}
conferir(all(r["produto_id"] in prod_ids for r in tables["fact_vendas"]), "todo produto da venda existe em dim_produtos")
conferir(all(r["dispositivo_id"] in maq_ids for r in tables["fact_vendas"]), "toda máquina da venda existe em dim_maquinas")
conferir(all(r["id_evento"] in ev_ids for r in tables["fact_vendas"]), "todo evento da venda existe em dim_evento")

# 3. A coluna status_venda veio preenchida?
# Se vier toda vazia, o notebook ainda consegue identificar cancelada/devolução pelo texto
# "Devolución de orden ...", mas é bom saber.
status = collections.Counter(str(r.get("status_venda")) for r in tables["fact_vendas"])
print("status_venda:", dict(status))
if status.get("None", 0) == len(tables["fact_vendas"]):
    print("AVISO — status_venda veio toda vazia. O notebook vai identificar cancelada/devolução pelo texto da devolução.")
else:
    conferir("Venda" in status, "status_venda tem vendas marcadas como Venda")

# 4. Quantos pedidos cancelados e devolvidos existem?
# Cancelada = venda original que foi cancelada. Devolução = estorno dela. Juntas, elas se anulam.
canceladas = [r for r in tables["fact_vendas"] if "cancel" in str(r.get("status_venda") or "").lower()]
devolucoes = [r for r in tables["fact_vendas"] if "devol" in str(r.get("status_venda") or "").lower()]
if not canceladas and not devolucoes:
    # Se a coluna veio vazia, procura pelo texto "Devolución de orden ..."
    devolucoes = [r for r in tables["fact_vendas"] if "devol" in str(r.get("metodo_pagamento") or "").lower()]
print(f"Canceladas: {len(canceladas)} linha(s) | Devoluções: {len(devolucoes)} linha(s)")

# 5. Pix e Dinheiro não devem ter bandeira
# Se um pagamento Pix vier com bandeira = "Pix", o gráfico ficaria "Pix / Pix". O cálculo corrige isso depois.
pix_com_bandeira = [r for r in tables["fact_pagamentos"] if str(r.get("metodo_pagamento")).lower() == "pix" and r.get("bandeira") not in (None, "", "None")]
print(f"Pagamentos Pix com bandeira preenchida (deveria ser 0): {len(pix_com_bandeira)}")

# 6. Receita e pedidos batem com o esperado?
# Soma por pedido, contando só vendas efetivas (sem cancelada/devolução).
pedidos_validos = {}
for r in tables["fact_vendas"]:
    st = str(r.get("status_venda") or "").lower()
    if "cancel" in st or "devol" in st:
        continue
    pedidos_validos.setdefault(r["numero_pedido"], to_num(r.get("total")))
receita = sum(pedidos_validos.values())
print(f"Receita (só vendas efetivas): R$ {receita:,.2f} | Pedidos válidos: {len(pedidos_validos)}")
conferir(receita > 0, "receita maior que zero")
conferir(len(pedidos_validos) > 0, "há pedidos válidos")

print("\nTodas as conferências passaram. Pode seguir para a próxima célula.")


In [ ]:
# CÉLULA 4 — Montar o JSON do site (data.json)
# O que esta célula faz: calcula os totais e monta o data.json com os campos que o site usa.
# Por que: o site não lê o .sql direto. Ele lê o data.json, que já vem com tudo calculado.
# Campos usados aqui (só deste projeto de vendas): receita, pedidos, itens, ticket médio, eventos,
# máquinas, produtos, pagamentos, vendas por evento, por produto, por categoria, por máquina,
# por pagamento, por hora e a lista de itens. Canceladas e devoluções ficam separadas, fora da receita.
import json
from pathlib import Path

def build_json(tables: dict, sample_n: int = 120) -> dict:
    evento = {r["id_evento"]: r for r in tables["dim_evento"]}
    prod = {r["produto_id"]: r for r in tables["dim_produtos"]}
    maq = {r["dispositivo_id"]: r for r in tables["dim_maquinas"]}
    _vendas_raw = tables["fact_vendas"]; _pag_raw = tables["fact_pagamentos"]

    # --- Devoluções / estornos / cancelamentos: NÃO contam como receita ---
    # Por que 3 jeitos de detectar? A coluna nova (status_venda) veio toda NULL neste dump,
    # então a devolução só aparece no texto "Devolución de orden 796085" e no total negativo.
    # Para o código continuar funcionando quando a coluna nova vier preenchida, testamos os 3.
    def _txt(v):
        return str(v or "").lower()
    def _eh_devolucao_venda(r):
        # 1) coluna nova, quando vier preenchida: status_venda = devolução / cancelado / estorno
        st = _txt(r.get("status_venda"))
        if any(k in st for k in ("devol", "cancel", "estorno")):
            return True
        # 2) texto no metodo_pagamento: "Devolución de orden 796085"
        if any(k in _txt(r.get("metodo_pagamento")) for k in ("devol", "cancel", "estorno")):
            return True
        # 3) total negativo = estorno
        try:
            if float(r.get("total") or 0) < 0:
                return True
        except (TypeError, ValueError):
            pass
        return False
    def _eh_devolucao_pag(r):
        if any(k in _txt(r.get("metodo_pagamento")) for k in ("devol", "cancel", "estorno")):
            return True
        if any(k in _txt(r.get("bandeira")) for k in ("devol", "cancel", "estorno")):
            return True
        try:
            if float(r.get("valor_pagamento") or 0) < 0:
                return True
        except (TypeError, ValueError):
            pass
        return False
    # --- Bandeira: Pix / Dinheiro NÃO têm bandeira ---
    # Se o dump vier com bandeira="Pix" para um pagamento Pix, forçamos para "—" (Sem bandeira),
    # para o gráfico "Método + bandeira" não mostrar "Pix / Pix".
    def _bandeira_limpa(r):
        metodo = _txt(r.get("metodo_pagamento"))
        band = str(r.get("bandeira") or "").strip()
        if band.lower() == "pix":
            return "—"
        if any(k in metodo for k in ("pix", "dinheiro", "dinero", "efectivo", "efectivo", "cash")):
            return "—"
        return band or "—"

    # Mapa pedido -> status: fact_pagamentos NÃO tem status_venda, então usamos o status da venda
    # para decidir se o pagamento dela também deve ser excluído.
    # Ex: pedido 796085 = Cancelada (venda original), 798405 = Devolução (estorno dela).
    # O pagamento de 796085 (Amex R$ 72) parece normal, mas pertence a uma venda cancelada,
    # então também sai dos gráficos de pagamento. Sem isso, receita e pagamento ficariam divergentes.
    _status_por_pedido = {}
    for r in _vendas_raw:
        _status_por_pedido.setdefault(r.get("numero_pedido"), _txt(r.get("status_venda")))
    # Se status_venda vier vazio (dump antigo), descobre a Cancelada pelo texto da Devolução:
    # "Devolución de orden 796085" -> o pedido 796085 é a venda original cancelada.
    for r in _vendas_raw:
        m = re.search(r"orden\s+(\d+)", _txt(r.get("metodo_pagamento")))
        if m and "devol" in _txt(r.get("metodo_pagamento")):
            try:
                _status_por_pedido[int(m.group(1))] = "cancelada (inferida pela devolução)"
            except ValueError:
                pass
    # E a função _eh_devolucao_venda precisa enxergar essa Cancelada inferida
    _orig_eh_dev = _eh_devolucao_venda
    def _eh_devolucao_venda(r, _orig=_orig_eh_dev):
        if _orig(r):
            return True
        return any(k in _status_por_pedido.get(r.get("numero_pedido"), "") for k in ("devol", "cancel", "estorno"))
    def _eh_cancelada_ou_devolucao_pag(r):
        if _eh_devolucao_pag(r):
            return True
        st = _status_por_pedido.get(r.get("numero_pedido"), "")
        return any(k in st for k in ("devol", "cancel", "estorno"))

    vendas_dev = [r for r in _vendas_raw if _txt(r.get("status_venda")).find("devol") >= 0 or _eh_devolucao_venda(r) and _txt(r.get("status_venda")).find("cancel") < 0]
    # Separa de verdade: Devolução vs Cancelada (pela coluna status_venda, que agora vem preenchida)
    vendas_devolucao = [r for r in _vendas_raw if "devol" in _txt(r.get("status_venda")) or "devol" in _txt(r.get("metodo_pagamento"))]
    vendas_cancelada = [r for r in _vendas_raw if "cancel" in _txt(r.get("status_venda")) or "cancel" in _status_por_pedido.get(r.get("numero_pedido"), "") and "devol" not in _txt(r.get("metodo_pagamento"))]
    # Fallback: se a coluna vier vazia num dump futuro, usa a detecção por texto/valor para Devolução
    if not vendas_devolucao and not vendas_cancelada:
        vendas_devolucao = [r for r in _vendas_raw if _eh_devolucao_venda(r)]
        vendas_cancelada = []
    vendas_dev = vendas_devolucao + vendas_cancelada
    pag_dev = [r for r in _pag_raw if _eh_cancelada_ou_devolucao_pag(r)]
    vendas = [r for r in _vendas_raw if not _eh_devolucao_venda(r)]
    pag = [r for r in _pag_raw if not _eh_cancelada_ou_devolucao_pag(r)]
    # KPIs separados: Devolução vs Cancelada
    # Devolução 798405 (-R$ 72) é o estorno da Cancelada 796085 (+R$ 72). Juntas se anulam,
    # por isso a receita final já está líquida e NÃO devemos somar a devolução de novo.
    dev_pedidos = {r.get("numero_pedido") for r in vendas_devolucao}
    can_pedidos = {r.get("numero_pedido") for r in vendas_cancelada}
    dev_count = len(dev_pedidos)
    can_count = len(can_pedidos)
    # Valor da devolução: pagamento de 798405 (-72). Valor da cancelada: total da venda 796085 (+72).
    dev_valor = round(sum(to_num(r.get("valor_pagamento")) for r in pag_dev if r.get("numero_pedido") in dev_pedidos), 2)
    if dev_valor == 0 and vendas_devolucao:
        dev_valor = round(sum(to_num(r.get("total")) for r in {r.get("numero_pedido"): r for r in vendas_devolucao}.values()), 2)
    can_valor = round(sum(to_num(r.get("total")) for r in {r.get("numero_pedido"): r for r in vendas_cancelada}.values()), 2)
    dev_itens = int(sum(to_num(r.get("quantidade"), 0) for r in vendas_devolucao))
    can_itens = int(sum(to_num(r.get("quantidade"), 0) for r in vendas_cancelada))


    orders = {}
    for r in vendas: orders.setdefault(r.get("numero_pedido"), to_num(r.get("total")))
    total_pedidos = len(orders); receita = sum(orders.values())
    receita_itens = sum(to_num(r.get("total_produto")) for r in vendas)
    qtd = sum(to_num(r.get("quantidade"), 0) for r in vendas)

    by_date = collections.defaultdict(lambda: {"revenue": 0.0, "orders": set(), "items": 0})
    for r in vendas:
        dt = (evento.get(r.get("id_evento")) or {}).get("data") or "sem data"
        by_date[dt]["revenue"] += to_num(r.get("total_produto"))
        by_date[dt]["orders"].add(r.get("numero_pedido"))
        by_date[dt]["items"] += to_num(r.get("quantidade"), 0)
    sales_by_date = [{"date": k, "revenue": round(v["revenue"], 2),
                      "orders": len(v["orders"]), "items": int(v["items"])}
                     for k, v in sorted(by_date.items())]

    def agg(key_fn):
        a = collections.defaultdict(lambda: {"revenue": 0.0, "quantity": 0, "orders": set()})
        for r in vendas:
            k = key_fn(r)
            a[k]["revenue"] += to_num(r.get("total_produto"))
            a[k]["quantity"] += to_num(r.get("quantidade"), 0)
            a[k]["orders"].add(r.get("numero_pedido"))
        return a

    bp = agg(lambda r: (prod.get(r.get("produto_id")) or {}).get("produto", "Desconhecido"))
    prod_by_name = {p.get("produto"): p for p in tables["dim_produtos"]}
    sales_by_product = sorted(
        [{"product": k, "category": prod_by_name.get(k, {}).get("categoria", ""),
          "revenue": round(v["revenue"], 2), "quantity": int(v["quantity"]), "orders": len(v["orders"])}
         for k, v in bp.items()], key=lambda x: -x["revenue"])

    bc = agg(lambda r: (prod.get(r.get("produto_id")) or {}).get("categoria", "Sem categoria"))
    sales_by_category = sorted(
        [{"category": k, "revenue": round(v["revenue"], 2),
          "quantity": int(v["quantity"]), "orders": len(v["orders"])}
         for k, v in bc.items()], key=lambda x: -x["revenue"])

    bm = agg(lambda r: r.get("dispositivo_id"))
    sales_by_machine = sorted(
        [{"dispositivo_id": k,
          "machine": f"Máquina {k}",
          "revenue": round(v["revenue"], 2), "quantity": int(v["quantity"]), "orders": len(v["orders"])}
         for k, v in bm.items()], key=lambda x: -x["revenue"])

    pp = collections.defaultdict(lambda: {"revenue": 0.0, "orders": set()})
    for r in pag:
        key = (r.get("metodo_pagamento"), _bandeira_limpa(r))
        pp[key]["revenue"] += to_num(r.get("valor_pagamento")); pp[key]["orders"].add(r.get("numero_pedido"))
    sales_by_payment = sorted(
        [{"method": k[0], "brand": k[1], "revenue": round(v["revenue"], 2), "orders": len(v["orders"])}
         for k, v in pp.items()], key=lambda x: -x["revenue"])

    pm = collections.defaultdict(lambda: {"revenue": 0.0, "orders": set()})
    for r in pag:
        pm[r.get("metodo_pagamento")]["revenue"] += to_num(r.get("valor_pagamento"))
        pm[r.get("metodo_pagamento")]["orders"].add(r.get("numero_pedido"))
    sales_by_payment_method = sorted(
        [{"method": k, "revenue": round(v["revenue"], 2), "orders": len(v["orders"])}
         for k, v in pm.items()], key=lambda x: -x["revenue"])

    by_hour = collections.defaultdict(lambda: {"revenue": 0.0, "orders": set()})
    for r in vendas:
        h = str(r.get("hora") or "")[:2] or "—"
        by_hour[h]["revenue"] += to_num(r.get("total_produto")); by_hour[h]["orders"].add(r.get("numero_pedido"))
    sales_by_hour = [{"hour": k, "revenue": round(v["revenue"], 2), "orders": len(v["orders"])}
                     for k, v in sorted(by_hour.items())]

    # --- Hora relativa ao início do evento: hora da venda - horário de início ---
    # Ex: evento começa 19:15, venda às 18:30 -> delta -45min -> bucket -1h (1h antes até o início)
    #     venda às 19:20 -> delta +5min -> bucket 0h (primeira hora do evento)
    # Usar hora relativa (e não hora do relógio) permite somar eventos que começam em horários diferentes.
    def _to_min(t):
        try:
            parts = str(t or "").split(":")
            return int(parts[0]) * 60 + int(parts[1])
        except (IndexError, ValueError, TypeError):
            return None
    by_rel = collections.defaultdict(lambda: {"revenue": 0.0, "orders": set(), "items": 0})
    for r in vendas:
        ev = evento.get(r.get("id_evento")) or {}
        start_min = _to_min(ev.get("horario_inicio"))
        sale_min = _to_min(r.get("hora"))
        if start_min is None or sale_min is None:
            continue
        delta = sale_min - start_min
        bucket = int(delta // 60)  # floor: -30min -> -1, 0-59min -> 0, 60-119min -> 1
        by_rel[bucket]["revenue"] += to_num(r.get("total_produto"))
        by_rel[bucket]["orders"].add(r.get("numero_pedido"))
        by_rel[bucket]["items"] += to_num(r.get("quantidade"), 0)
    def _rel_label(b):
        if b == 0: return "0h (início)"
        if b < 0: return f"{b}h"
        return f"+{b}h"
    sales_by_relative_hour = [{"relative_hour": b, "label": _rel_label(b),
                               "revenue": round(v["revenue"], 2), "orders": len(v["orders"]), "items": int(v["items"])}
                              for b, v in sorted(by_rel.items())]

    events = []
    for ev in sorted(tables["dim_evento"], key=lambda x: x.get("data") or ""):
        rows = [r for r in vendas if r.get("id_evento") == ev.get("id_evento")]
        events.append({"id_evento": ev.get("id_evento"), "date": ev.get("data"),
                       "horario_inicio": ev.get("horario_inicio"), "tipo_evento": ev.get("tipo_evento"),
                       "revenue": round(sum(to_num(r.get("total_produto")) for r in rows), 2),
                       "orders": len({r.get("numero_pedido") for r in rows}),
                       "items": int(sum(to_num(r.get("quantidade"), 0) for r in rows))})

    sample = []
    for r in vendas[:sample_n]:
        p = prod.get(r.get("produto_id")) or {}; ev = evento.get(r.get("id_evento")) or {}
        sample.append({"numero_pedido": r.get("numero_pedido"), "date": ev.get("data"),
                       "hora": r.get("hora"), "produto": p.get("produto"), "categoria": p.get("categoria"),
                       "quantidade": r.get("quantidade"), "preco_unitario": r.get("preco_unitario"),
                       "total_produto": r.get("total_produto"), "total_pedido": r.get("total"),
                       "metodo_pagamento": r.get("metodo_pagamento"),
                       "dispositivo_id": r.get("dispositivo_id"), "evento": ev.get("tipo_evento"),
                       "nome_comercio": r.get("nome_comercio") or "—"})

    # --- Breakdown por evento (filtro por evento/data nos gráficos) ---
    order_to_event = {}
    for r in vendas:
        order_to_event.setdefault(r.get("numero_pedido"), r.get("id_evento"))
    by_event = {}
    for ev in sorted(tables["dim_evento"], key=lambda x: x.get("data") or ""):
        eid = ev.get("id_evento"); dt = ev.get("data")
        ev_vendas = [r for r in vendas if r.get("id_evento") == eid]
        ev_orders = {r.get("numero_pedido") for r in ev_vendas}
        ev_pag = [r for r in pag if r.get("numero_pedido") in ev_orders]
        def agg_ev(rows, key_fn):
            a = collections.defaultdict(lambda: {"revenue": 0.0, "quantity": 0, "orders": set()})
            for r in rows:
                k = key_fn(r)
                a[k]["revenue"] += to_num(r.get("total_produto"))
                a[k]["quantity"] += to_num(r.get("quantidade"), 0)
                a[k]["orders"].add(r.get("numero_pedido"))
            return a
        bp_ev = agg_ev(ev_vendas, lambda r: (prod.get(r.get("produto_id")) or {}).get("produto", "Desconhecido"))
        bc_ev = agg_ev(ev_vendas, lambda r: (prod.get(r.get("produto_id")) or {}).get("categoria", "Sem categoria"))
        bm_ev = agg_ev(ev_vendas, lambda r: r.get("dispositivo_id"))
        pp_ev = collections.defaultdict(lambda: {"revenue": 0.0, "orders": set()})
        for r in ev_pag:
            key = (r.get("metodo_pagamento"), _bandeira_limpa(r))
            pp_ev[key]["revenue"] += to_num(r.get("valor_pagamento")); pp_ev[key]["orders"].add(r.get("numero_pedido"))
        pm_ev = collections.defaultdict(lambda: {"revenue": 0.0, "orders": set()})
        for r in ev_pag:
            pm_ev[r.get("metodo_pagamento")]["revenue"] += to_num(r.get("valor_pagamento"))
            pm_ev[r.get("metodo_pagamento")]["orders"].add(r.get("numero_pedido"))
        by_event[dt] = {
            "event_id": eid, "date": dt, "tipo_evento": ev.get("tipo_evento"),
            "revenue": round(sum(to_num(r.get("total_produto")) for r in ev_vendas), 2),
            "orders": len(ev_orders),
            "sales_by_product": sorted([{"product": k, "category": prod_by_name.get(k, {}).get("categoria", ""), "revenue": round(v["revenue"], 2), "quantity": int(v["quantity"]), "orders": len(v["orders"])} for k, v in bp_ev.items()], key=lambda x: -x["revenue"]),
            "sales_by_category": sorted([{"category": k, "revenue": round(v["revenue"], 2), "quantity": int(v["quantity"]), "orders": len(v["orders"])} for k, v in bc_ev.items()], key=lambda x: -x["revenue"]),
            "sales_by_machine": sorted([{"dispositivo_id": k, "machine": f"Máquina {k}", "revenue": round(v["revenue"], 2), "quantity": int(v["quantity"]), "orders": len(v["orders"])} for k, v in bm_ev.items()], key=lambda x: -x["revenue"]),
            "sales_by_payment": sorted([{"method": k[0], "brand": k[1], "revenue": round(v["revenue"], 2), "orders": len(v["orders"])} for k, v in pp_ev.items()], key=lambda x: -x["revenue"]),
            "sales_by_payment_method": sorted([{"method": k, "revenue": round(v["revenue"], 2), "orders": len(v["orders"])} for k, v in pm_ev.items()], key=lambda x: -x["revenue"]),
        }

    # --- Base completa (todos os itens de todos os pedidos) ---
    # O site pagina em 50 por vez, sem renderizar tudo — por isso não pesa na tela.
    full = []
    for r in vendas:
        pp = prod.get(r.get("produto_id")) or {}; evv = evento.get(r.get("id_evento")) or {}
        full.append({"numero_pedido": r.get("numero_pedido"), "date": evv.get("data"),
                     "hora": r.get("hora"), "produto": pp.get("produto"), "categoria": pp.get("categoria"),
                     "quantidade": r.get("quantidade"), "preco_unitario": r.get("preco_unitario"),
                     "total_produto": r.get("total_produto"), "total_pedido": r.get("total"),
                     "metodo_pagamento": r.get("metodo_pagamento"),
                     "dispositivo_id": r.get("dispositivo_id"), "evento": evv.get("tipo_evento"), "nome_comercio": r.get("nome_comercio") or "—"})

    # --- Breakdown por comércio (filtro por comércio nos gráficos) ---
    # Agrupa por nome_comercio. Inclui produtos e hora relativa por comércio.
    def _to_min2(t):
        try:
            parts = str(t or "").split(":")
            return int(parts[0]) * 60 + int(parts[1])
        except (IndexError, ValueError, TypeError):
            return None
    comercios = sorted({(r.get("nome_comercio") or "—") for r in vendas})
    by_comercio = {}
    sales_by_comercio = []
    for com in comercios:
        com_vendas = [r for r in vendas if (r.get("nome_comercio") or "—") == com]
        com_orders = {r.get("numero_pedido") for r in com_vendas}
        # Produtos por comércio
        bp_com = collections.defaultdict(lambda: {"revenue": 0.0, "quantity": 0, "orders": set()})
        for r in com_vendas:
            pname = (prod.get(r.get("produto_id")) or {}).get("produto", "Desconhecido")
            bp_com[pname]["revenue"] += to_num(r.get("total_produto"))
            bp_com[pname]["quantity"] += to_num(r.get("quantidade"), 0)
            bp_com[pname]["orders"].add(r.get("numero_pedido"))
        # Hora relativa por comércio
        by_rel_com = collections.defaultdict(lambda: {"revenue": 0.0, "orders": set(), "items": 0})
        for r in com_vendas:
            ev = evento.get(r.get("id_evento")) or {}
            sm = _to_min2(ev.get("horario_inicio")); hm = _to_min2(r.get("hora"))
            if sm is None or hm is None: continue
            b = int((hm - sm) // 60)
            by_rel_com[b]["revenue"] += to_num(r.get("total_produto"))
            by_rel_com[b]["orders"].add(r.get("numero_pedido"))
            by_rel_com[b]["items"] += to_num(r.get("quantidade"), 0)
        def _rel_label2(b):
            if b == 0: return "0h (início)"
            if b < 0: return f"{b}h"
            return f"+{b}h"
        prod_by_name2 = prod_by_name
        sales_by_product_com = sorted([{"product": k, "category": prod_by_name2.get(k, {}).get("categoria", ""), "revenue": round(v["revenue"], 2), "quantity": int(v["quantity"]), "orders": len(v["orders"])} for k, v in bp_com.items()], key=lambda x: -x["revenue"])
        rel_com = [{"relative_hour": b, "label": _rel_label2(b), "revenue": round(v["revenue"], 2), "orders": len(v["orders"]), "items": int(v["items"])} for b, v in sorted(by_rel_com.items())]
        rev_com = round(sum(to_num(r.get("total_produto")) for r in com_vendas), 2)
        by_comercio[com] = {"comercio": com, "revenue": rev_com, "orders": len(com_orders), "items": int(sum(to_num(r.get("quantidade"), 0) for r in com_vendas)), "sales_by_product": sales_by_product_com, "sales_by_relative_hour": rel_com}
        sales_by_comercio.append({"comercio": com, "revenue": rev_com, "orders": len(com_orders), "items": int(sum(to_num(r.get("quantidade"), 0) for r in com_vendas))})
    sales_by_comercio = sorted(sales_by_comercio, key=lambda x: -x["revenue"])

    return {
        "meta": {
            "project": "Sales Data Warehouse",
            "architecture": "Medallion (Bronze → Silver → Gold) • SQL Server (SSMS)",
            "source": "gold — dim_evento, dim_maquinas, dim_produtos, fact_vendas, fact_pagamentos",
            "currency": "BRL",
            "privacy_note": "Campos sensíveis removidos: identificacao_comprador, ultimos_digitos_cartao, codigo_serie.",
            "devolucoes_note": "Devoluções / estornos / cancelados excluídos da receita e contados à parte em kpis.devolucoes_*.",
            "last_updated": datetime.now().strftime("%d/%m/%Y"),
        },
        "kpis": {
            "total_revenue": round(receita, 2), "revenue_by_items": round(receita_itens, 2),
            "total_orders": total_pedidos, "total_items": int(qtd),
            "avg_ticket": round(receita / total_pedidos, 2) if total_pedidos else 0,
            "events_count": len(evento),
            "active_machines": len({r.get("dispositivo_id") for r in vendas}),
            "products_count": len({r.get("produto_id") for r in vendas}),
            "payment_records": len(pag),
            "devolucoes_count": dev_count, "devolucoes_valor": dev_valor, "devolucoes_itens": dev_itens,
            "canceladas_count": can_count, "canceladas_valor": can_valor, "canceladas_itens": can_itens,
            "canceladas_exemplo": sorted(can_pedidos)[0] if can_pedidos else None, "devolucoes_exemplo": sorted(dev_pedidos)[0] if dev_pedidos else None,
        },
        "sales_by_date": sales_by_date, "sales_by_product": sales_by_product,
        "sales_by_category": sales_by_category, "sales_by_machine": sales_by_machine,
        "sales_by_payment": sales_by_payment, "sales_by_payment_method": sales_by_payment_method,
        "sales_by_hour": sales_by_hour, "sales_by_relative_hour": sales_by_relative_hour, "sales_by_comercio": sales_by_comercio, "events": events, "fact_sales_sample": sample, "fact_sales_all": full, "by_event": by_event, "by_comercio": by_comercio,
    }

PLACEHOLDER_COMMENT = "<!-- Dados carregados de data.json (versão web, sem cópia embutida para ficar leve) -->"

# --- Executa o cálculo ---
data = build_json(tables)
Path("/content/data.json").write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
print("OK! data.json gerado em /content/data.json")
print(f"Receita: R$ {data['kpis']['total_revenue']:,.2f} | Pedidos: {data['kpis']['total_orders']} | Eventos: {data['kpis']['events_count']}")
print(f"Canceladas: {data['kpis'].get('canceladas_count', 0)} ({data['kpis'].get('canceladas_valor', 0)}) | Devoluções: {data['kpis'].get('devolucoes_count', 0)} ({data['kpis'].get('devolucoes_valor', 0)})")


In [ ]:
# CÉLULA 5 — Gerar index.html (leve) + index-standalone.html (1 arquivo só)
# O que esta célula faz: pede para você subir o index.html atual e gera as 2 versões do site
# a partir do mesmo data.json, com os mesmos dados, gráficos e visual.
# Por que 2 versões:
# - Leve: sem dados dentro, busca o data.json com fetch(). Para GitHub Pages / servidor.
#   Não funciona com dois cliques, porque o navegador bloqueia fetch() em file://.
# - Standalone: com os dados dentro, 1 arquivo só. Para mandar por WhatsApp / e-mail e abrir com dois cliques.
# A única diferença entre elas é onde o dado mora. O resto é igual.
from google.colab import files
import re, json
from pathlib import Path

print("Suba o index.html atual (pode ser a versão leve ou a standalone, tanto faz)...")
up = files.upload()
html_nome = next((n for n in up.keys() if n.lower().endswith(".html")), None)
if html_nome is None:
    raise ValueError("Nenhum arquivo .html encontrado. Suba o index.html atual.")
html = up[html_nome].decode("utf-8")
print(f"Arquivo recebido: {html_nome} ({len(html)} caracteres)")

# --- Tirar os dados embutidos, contando chaves ---
# Por que contar chaves e não cortar no primeiro "};": o JSON tem muitas chaves dentro dele.
# Se cortar no primeiro "};", sobra um pedaço do JSON no HTML e o arquivo fica quebrado.
# Contando chaves, a gente acha o fim verdadeiro do JSON, mesmo que ele tenha "};" dentro de textos.
def tirar_embutido_por_chaves(html_text: str) -> str:
    # Procura a tag <script id="embedded-data">
    m = re.search(r'<script id="embedded-data"[^>]*>', html_text)
    if not m:
        return html_text
    # Acha a primeira chave { depois da tag
    start = html_text.find("{", m.end())
    if start == -1:
        return html_text
    # Conta chaves, respeitando textos entre aspas
    depth = 0; i = start; em_aspas = False; esc = False
    while i < len(html_text):
        c = html_text[i]
        if em_aspas:
            if esc: esc = False
            elif c == "\\": esc = True
            elif c == '"': em_aspas = False
        else:
            if c == '"': em_aspas = True
            elif c == "{": depth += 1
            elif c == "}":
                depth -= 1
                if depth == 0:
                    break
        i += 1
    # Acha o </script> depois do fim do JSON
    fim_script = html_text.find("</script>", i)
    if fim_script == -1:
        return html_text
    # Troca tudo (da tag até o </script>) pelo comentário que marca a versão leve
    comentario = "<!-- Dados carregados de data.json (versão web, sem cópia embutida para ficar leve) -->"
    return html_text[:m.start()] + comentario + html_text[fim_script + len("</script>"):]

# --- Formato muito antigo: se o HTML tiver "var Ac" ou "EMBEDDED_DATA = {...}" ---
# É o mesmo raciocínio: acha o primeiro "{" depois da atribuição e conta chaves até o fim.
def tirar_formato_antigo(html_text: str) -> str:
    for padrao in [r"var\s+Ac\s*=", r"const\s+Ac\s*=", r"EMBEDDED_DATA\s*="]:
        m = re.search(padrao, html_text)
        if not m:
            continue
        start = html_text.find("{", m.end())
        if start == -1:
            continue
        depth = 0; i = start; em_aspas = False; esc = False
        while i < len(html_text):
            c = html_text[i]
            if em_aspas:
                if esc: esc = False
                elif c == "\\": esc = True
                elif c == '"': em_aspas = False
            else:
                if c == '"': em_aspas = True
                elif c == "{": depth += 1
                elif c == "}":
                    depth -= 1
                    if depth == 0:
                        break
            i += 1
        # Remove do início da linha até depois do "};" (se houver)
        inicio_linha = html_text.rfind("\n", 0, m.start()) + 1
        fim = i + 1
        if html_text[fim:fim+1] == ";":
            fim += 1
        comentario = "<!-- Dados carregados de data.json (versão web, sem cópia embutida para ficar leve) -->"
        return html_text[:inicio_linha] + comentario + html_text[fim:]
    return html_text

# --- Monta a versão leve ---
base = tirar_embutido_por_chaves(html)
base = tirar_formato_antigo(base)
COMENTARIO = "<!-- Dados carregados de data.json (versão web, sem cópia embutida para ficar leve) -->"
if "id=\"embedded-data\"" in base:
    raise ValueError("A versão leve ainda tem a tag embedded-data. Confira o HTML enviado.")
if COMENTARIO not in base:
    # Se o HTML não tinha nenhuma marca, insere o comentário antes de </body>
    base = base.replace("</body>", COMENTARIO + "\n</body>", 1) if "</body>" in base else base + "\n" + COMENTARIO

# --- Garante que o upload manual do JSON (filePicker) existe ---
# Se o HTML já tiver o campo, não mexe. Se não tiver, insere dentro da caixa de aviso.
if 'id="filePicker"' not in base:
    # Procura a caixa de aviso e insere o campo depois do título dela
    if 'id="loadBox"' in base:
        base = base.replace(
            '<h2>Não consegui carregar o data.json automaticamente</h2>',
            '<h2>Não consegui carregar o data.json automaticamente</h2>\n<p>Último recurso: selecione o data.json abaixo:</p>\n<p><input type="file" id="filePicker" accept=".json,application/json" style="background:#fff;color:#111"></p>',
            1,
        )
    print("AVISO — o HTML não tinha o campo de upload manual. O campo foi inserido na caixa de aviso.")
else:
    print("OK — o campo de upload manual (filePicker) já existe no HTML.")

# --- Gera as 2 versões ---
data = json.loads(Path("/content/data.json").read_text(encoding="utf-8"))
minificado = json.dumps(data, ensure_ascii=False, separators=(",", ":"))
tag = '<script id="embedded-data" type="application/json">\n' + minificado + '\n</script>'

leve = base
standalone = base.replace(COMENTARIO, tag, 1)
if COMENTARIO in standalone:
    raise ValueError("A standalone ainda tem o comentário da leve. Confira o HTML enviado.")

Path("/content/index.html").write_text(leve, encoding="utf-8")
Path("/content/index-standalone.html").write_text(standalone, encoding="utf-8")

# --- Conferências finais ---
print(f"\nVersão leve (index.html): {len(leve)/1024:.0f} KB — tem embedded-data? {'id=\"embedded-data\"' in leve} (deveria ser False)")
print(f"Versão standalone (index-standalone.html): {len(standalone)/1024/1024:.2f} MB — tem embedded-data? {'id=\"embedded-data\"' in standalone} (deveria ser True)")
print(f"filePicker nas 2 versões? Leve: {'id=\"filePicker\"' in leve} | Standalone: {'id=\"filePicker\"' in standalone}")
print(f"KPIs conferidos: Receita R$ {data['kpis']['total_revenue']:,.2f} | Pedidos {data['kpis']['total_orders']} | Eventos {data['kpis']['events_count']}")


In [ ]:
# CÉLULA 6 — Baixar os arquivos atualizados
# O que esta célula faz: baixa os 3 arquivos para o seu computador.
# Por que: o Colab guarda os arquivos só aqui dentro. Para usar no GitHub Pages ou mandar para alguém,
# você precisa baixar.
from google.colab import files

for nome in ["data.json", "index.html", "index-standalone.html"]:
    print(f"Baixando {nome}...")
    files.download(f"/content/{nome}")
print("\nPronto! Guarde os 3 na mesma pasta quando for usar a versão leve.")
print("Para a versão leve: index.html + data.json juntos, abertos com servidor / GitHub Pages.")
print("Para abrir com dois cliques: index-standalone.html sozinho.")
